<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Combine-annotations-into-one-csv-file" data-toc-modified-id="Combine-annotations-into-one-csv-file-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Combine annotations into one csv file</a></span><ul class="toc-item"><li><span><a href="#Select-folder-that-contains-the-annotation-files" data-toc-modified-id="Select-folder-that-contains-the-annotation-files-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Select folder that contains the annotation files</a></span></li><li><span><a href="#Combine-the-files-into-one-dataframes" data-toc-modified-id="Combine-the-files-into-one-dataframes-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>Combine the files into one dataframes</a></span></li></ul></li><li><span><a href="#Check-validity-of-various-columns" data-toc-modified-id="Check-validity-of-various-columns-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Check validity of various columns</a></span><ul class="toc-item"><li><span><a href="#Review-null-deployments:" data-toc-modified-id="Review-null-deployments:-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Review null deployments:</a></span></li><li><span><a href="#Check-species-names" data-toc-modified-id="Check-species-names-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Check species names</a></span></li><li><span><a href="#Check-MaxIterval-&amp;-TimeOfMax" data-toc-modified-id="Check-MaxIterval-&amp;-TimeOfMax-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Check MaxIterval &amp; TimeOfMax</a></span></li><li><span><a href="#Compare-SurveyID-presence-in-annotation-vs-metadata" data-toc-modified-id="Compare-SurveyID-presence-in-annotation-vs-metadata-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Compare SurveyID presence in annotation vs metadata</a></span></li><li><span><a href="#Review-duplicates" data-toc-modified-id="Review-duplicates-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Review duplicates</a></span></li></ul></li><li><span><a href="#Export-combined_df-to-combined-annotations-file" data-toc-modified-id="Export-combined_df-to-combined-annotations-file-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Export combined_df to combined annotations file</a></span></li></ul></div>

In [ ]:
# Last changed 2025.01.28

# Combine annotations into one csv file

This notebooks is part of the 2024/2025 Spyfish data cleaning efforts and is used to extract the combine all the files containing extracted expert annotations. This file will be uploaded to the S3 bucket.

In the second part of the notebook, there are some visual checks to see if something is suspicious with the annotations. These output should be ready for upload, so there shouldn't be any irregularities and if there are, it means that the previous notebook (Fix files and extract annotations) needs to be updated and the export file re-run - or potentially done by hand, but with a TODO in the above notebook.

TODO: Some of these checks could be repurposed for the automatic tests of the annotations being saved in the S3 buckets in the future.

In [ ]:
import os
import pandas as pd
import numpy as np
import datetime

from ipyfilechooser import FileChooser
from IPython.display import display

## Select folder that contains the annotation files

In [ ]:
folder_chooser = FileChooser(title=f'<b>Select the folder containing annotation files</b>')
display(folder_chooser)

In [ ]:
selected_folder = folder_chooser.selected_path
assert selected_folder != None, "Select folder in the cell above."
print(f"The selected folder is {selected_folder}")
all_files = os.listdir(selected_folder)
all_tab_files = [os.path.join(selected_folder, file) for file in all_files if file.endswith(".csv")]
all_tab_files

## Combine the files into one dataframes


In [ ]:
def combine_annotations(all_tab_files):
    dfs = []
    # review lines test to see if all the lines were saved in the dataframe
    check_lines = 0
    for f in all_tab_files:
        try: 
            data = pd.read_csv(f)

            dfs.append(data)
            check_lines += data.shape[0]
        except:
            print(f"{f} not read")

    combined_df = pd.concat(dfs, axis=0)
    assert combined_df.shape[0] == check_lines
    return combined_df

In [ ]:
combined_df = combine_annotations(all_tab_files)
combined_df.shape, combined_df.columns

In [ ]:
# TODO: this step should happen in the Fix files and extract annotation notebook 

combined_df['ScientificName'] = combined_df['ScientificName'].fillna('NULL')
combined_df['TimeOfMax'] = combined_df['TimeOfMax'].fillna('NULL')
combined_df['MaxInterval'] = combined_df['MaxInterval'].fillna('NULL')
combined_df['ConfidenceAgreement'] = combined_df['ConfidenceAgreement'].fillna('NA')

In [ ]:
combined_df

In [ ]:
combined_df.sample(20)

# Check validity of various columns

## Review null deployments:


In [ ]:
# combined_df[combined_df["ScientificName"] == "NULL"]

In [ ]:
# Repeating DeploymentID with Null deployment, should only be one
# TODO Review this when sites and replicate within sites are fixed

null_dep = combined_df[combined_df["ScientificName"] == "NULL"].groupby("DeploymentID").count()
null_dep[null_dep["ScientificName"] > 1]

## Check species names

Species underscored with FIX_ need review, as do sp1, sp2, sp3, sp4, sp5, sp6, sp7, as do any mention of unknown/undefined.

In [ ]:
combined_df[combined_df["ScientificName"].isna()]

In [ ]:
species_names = combined_df["ScientificName"].unique()
for i in np.sort(species_names[1:]):
    print(i)    

TODO:
- check species with species name checker to make sure all is good


## Check MaxIterval & TimeOfMax

In [ ]:
combined_df["MaxInterval"].unique()

In [ ]:
# Are there any times that do not follow the predefined format or NULL
# TODO: now this checks that the string is 8 long, it would be good to check with a regex str
combined_df[(combined_df["TimeOfMax"].str.len() != 8) & (combined_df["TimeOfMax"] != 'NULL')]

## Compare SurveyID presence in annotation vs metadata 

In [ ]:
surveyIDs_annotations = combined_df['DeploymentID'].str[:16].unique()
surveyIDs_annotations, len(surveyIDs_annotations)

In [ ]:
# Surveys present in Survey metadata file 
# (Copy survey list)
# TODO, check this, a way to get this automatically from the sharepoint lists?

survey_text = """ABC_20200220_BUV
EFG_20200220_BUV"""
surveyIDs_metadata = survey_text.split("\n")
print(surveyIDs_metadata)

In [ ]:
common = set(surveyIDs_annotations) & set(surveyIDs_metadata)
only_in_annotations = set(surveyIDs_annotations) - set(surveyIDs_metadata)
only_in_surveys = set(surveyIDs_metadata) - set(surveyIDs_annotations)
common_list = sorted(list(common))
only_in_annotations_list = sorted(list(only_in_annotations))
only_in_surveys_list = sorted(list(only_in_surveys))

print(f"Reviewing files annotations and surveys, there are {len(common)} SurveyIDs in common." )
print(f"The two files have the following {len(common)} SurveyIds in common:")
print(common_list)
print(f"The {len(only_in_annotations)} SurveyIDs present only in annotations are:")
print(only_in_annotations_list)
print(f"The {len(only_in_surveys)} SurveyIDs present only in surveys are:")
print(only_in_surveys_list)

In [ ]:
# Find presence of specific SurveyIDs
combined_df[combined_df['DeploymentID'].str[:16].isin(['ABC_20200220_BUV', 'EFG_20200220_BUV'])]

## Review duplicates

In [ ]:
# TODO: why are there duplicates???? Review when ReplicateWithinSite is reviewed
combined_df[combined_df.duplicated(keep=False)]

# Export combined_df to combined annotations file

In [ ]:
# Get the current date for the annotations file
current_date = str(datetime.date.today())
current_date

In [ ]:
# Create export folder in folder containing the annotations folder

path_to_export = os.path.join(selected_folder, "export")
os.makedirs(path_to_export, exist_ok=True)


export_excel_file_name = f"{current_date}_annotations_buv_doc_combined.csv"
export_location = os.path.join(path_to_export, export_excel_file_name)

print(f"File containing the conacatenated annotations exported to: '{export_location}'")
combined_df.to_csv(export_location)  